In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

backend = BasicSimulator()

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.


## Quantum Random Number Generator

In [ ]:
def quantum_random_bits(n):
    """Generates n random bits using quantum measurement of the |+> state."""
    bits = []
    for _ in range(n):
        qc = QuantumCircuit(1, 1)
        qc.h(0)
        qc.measure(0, 0)
        t_qc = transpile(qc, backend)
        result = backend.run(t_qc, shots=1).result()
        bits.append(int(list(result.get_counts().keys())[0]))
    return bits

## Alice's Part

In [ ]:
NUM_QUBITS = 40
alice_bits = quantum_random_bits(NUM_QUBITS)
alice_bases = quantum_random_bits(NUM_QUBITS)

def alice_encode(bit, basis):
    qc = QuantumCircuit(1)
    if bit == 1: qc.x(0)
    if basis == 1: qc.h(0)
    return qc

alice_qubits = [alice_encode(alice_bits[i], alice_bases[i]) for i in range(NUM_QUBITS)]
print(f"Alice prepared {NUM_QUBITS} qubits.")

## Bob's Part

In [ ]:
bob_bases = quantum_random_bits(NUM_QUBITS)

def bob_measure(qubit_qc, basis):
    qc = qubit_qc.copy()
    if basis == 1: qc.h(0)
    qc.add_register(__import__('qiskit').ClassicalRegister(1))
    qc.measure(0, 0)
    t_qc = transpile(qc, backend)
    return int(list(backend.run(t_qc, shots=1).result().get_counts().keys())[0])

bob_bits = [bob_measure(alice_qubits[i], bob_bases[i]) for i in range(NUM_QUBITS)]

## Sifting

In [ ]:
matching_indices = [i for i in range(NUM_QUBITS) if alice_bases[i] == bob_bases[i]]
alice_key = [alice_bits[i] for i in matching_indices]
bob_key = [bob_bits[i] for i in matching_indices]

print(f"Keys match: {alice_key == bob_key}")
print(f"Key: {alice_key}")